# 1) Exercício: Regra de Associação Algoritmo Apriori

In [60]:
#Instalado já a biblioteca 

#!pip install apyori
import os
import pandas as pd

from apyori import apriori

In [61]:
#!pip install kagglehub


In [62]:

# para acessar os dados da kaggle
import kagglehub



# Base de dados Market Basket Analysis

## Download e preparação dos dados

Desta vez, você vai utilizar uma base de dados disponível no Kaggle, que contém um histórico de aquisição de itens em uma loja. Cada linha representa a transação de um item, com o ID da compra, quantidade, data, preço, ID do cliente e país.

https://www.kaggle.com/datasets/aslanahmedov/market-basket-analysis

> * Número de Linhas: 522065
> * Número de Atributos: 7
>   * BillNo: Número de 6 dígitos atribuído a cada transação. Nominal.
>   * Itemname: Nome do produto. Nominal.
>   * Quantity: As quantidades de cada produto por transação. Numérico.
>   * Date: O dia e a hora em que cada transação foi gerada. Numérico.
>   * Price: Preço do produto. Numérico.
>   * CustomerID: Número de 5 dígitos atribuído a cada cliente. Nominal.
>   * Country: Nome do país onde cada cliente reside. Nominal.


In [63]:
path = kagglehub.dataset_download("aslanahmedov/market-basket-analysis")
# acesso a base de dados.
print("Path to dataset files:", path)

Path to dataset files: /home/fabiene/.cache/kagglehub/datasets/aslanahmedov/market-basket-analysis/versions/2


In [64]:
!ls {path}

Assignment-1_Data.csv  Assignment-1_Data.xlsx


no linux:

(base) :~/.cache/kagglehub/datasets/aslanahmedov/market-basket-analysis$ ls -ltr

total 4
drwxrwxr-x 3 fabiene fabiene 4096 ago 10 16:48 versions

-rw-rw-r-- 1 fabiene fabiene    0 ago 10 16:48 2.complete

(base) :~/.cache/kagglehub/datasets/aslanahmedov/market-basket-analysis$ 



no linux

(base) fabiene@xxxxx:~/.cache/kagglehub/datasets/aslanahmedov/market-basket-analysis/versions/2$ ls -ltr

total 57968

-rw-rw-r-- 1 fabiene fabiene 41204583 ago 10 16:48 Assignment-1_Data.csv

-rw-rw-r-- 1 fabiene fabiene 18146884 ago 10 16:48 Assignment-1_Data.xlsx

(base) fabiene@xxxxxx:~/.cache/kagglehub/datasets/aslanahmedov/market-basket-analysis/versions/2$ 


O dataset é fornecido em formato `csv` e `xlsx` (Excel). Nós vamos utilizar o arquivo `csv`, então vamos inspecionar suas primeiras linhas.

In [65]:
# ecolhido coletar os dados em .csv
!head {path}/Assignment-1_Data.csv

﻿BillNo;Itemname;Quantity;Date;Price;CustomerID;Country
536365;WHITE HANGING HEART T-LIGHT HOLDER;6;01.12.2010 08:26;2,55;17850;United Kingdom
536365;WHITE METAL LANTERN;6;01.12.2010 08:26;3,39;17850;United Kingdom
536365;CREAM CUPID HEARTS COAT HANGER;8;01.12.2010 08:26;2,75;17850;United Kingdom
536365;KNITTED UNION FLAG HOT WATER BOTTLE;6;01.12.2010 08:26;3,39;17850;United Kingdom
536365;RED WOOLLY HOTTIE WHITE HEART.;6;01.12.2010 08:26;3,39;17850;United Kingdom
536365;SET 7 BABUSHKA NESTING BOXES;2;01.12.2010 08:26;7,65;17850;United Kingdom
536365;GLASS STAR FROSTED T-LIGHT HOLDER;6;01.12.2010 08:26;4,25;17850;United Kingdom
536366;HAND WARMER UNION JACK;6;01.12.2010 08:28;1,85;17850;United Kingdom
536366;HAND WARMER RED POLKA DOT;6;01.12.2010 08:28;1,85;17850;United Kingdom


O arquivo contém um cabeçalho, e os campos são separados por `;`. Então, vamos carregar no Python levando isso em consideração.

In [66]:
mba = pd.read_csv(os.path.join(path, "Assignment-1_Data.csv"), delimiter=";")
mba.tail()

/tmp/ipykernel_5959/156717607.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  mba = pd.read_csv(os.path.join(path, "Assignment-1_Data.csv"), delimiter=";")


,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
522059,581587,PACK OF 20 SPACEBOY NAPKINS,12,09.12.2011 12:50,"0,85",12680.0,France
522060,581587,CHILDREN'S APRON DOLLY GIRL,6,09.12.2011 12:50,"2,1",12680.0,France
522061,581587,CHILDRENS CUTLERY DOLLY GIRL,4,09.12.2011 12:50,"4,15",12680.0,France
522062,581587,CHILDRENS CUTLERY CIRCUS PARADE,4,09.12.2011 12:50,"4,15",12680.0,France
522063,581587,BAKING SET 9 PIECE RETROSPOT,3,09.12.2011 12:50,"4,95",12680.0,France


Perceba que são mais de 500 mil transações.

a coluna Itemname é a lista dos produtos 

In [67]:
list(mba['Itemname'].unique())

['WHITE HANGING HEART T-LIGHT HOLDER',
 'WHITE METAL LANTERN',
 'CREAM CUPID HEARTS COAT HANGER',
 'KNITTED UNION FLAG HOT WATER BOTTLE',
 'RED WOOLLY HOTTIE WHITE HEART.',
 'SET 7 BABUSHKA NESTING BOXES',
 'GLASS STAR FROSTED T-LIGHT HOLDER',
 'HAND WARMER UNION JACK',
 'HAND WARMER RED POLKA DOT',
 'ASSORTED COLOUR BIRD ORNAMENT',
 "POPPY'S PLAYHOUSE BEDROOM",
 "POPPY'S PLAYHOUSE KITCHEN",
 'FELTCRAFT PRINCESS CHARLOTTE DOLL',
 'IVORY KNITTED MUG COSY',
 'BOX OF 6 ASSORTED COLOUR TEASPOONS',
 'BOX OF VINTAGE JIGSAW BLOCKS',
 'BOX OF VINTAGE ALPHABET BLOCKS',
 'HOME BUILDING BLOCK WORD',
 'LOVE BUILDING BLOCK WORD',
 'RECIPE BOX WITH METAL HEART',
 'DOORMAT NEW ENGLAND',
 'JAM MAKING SET WITH JARS',
 'RED COAT RACK PARIS FASHION',
 'YELLOW COAT RACK PARIS FASHION',
 'BLUE COAT RACK PARIS FASHION',
 'BATH BUILDING BLOCK WORD',
 'ALARM CLOCK BAKELIKE PINK',
 'ALARM CLOCK BAKELIKE RED',
 'ALARM CLOCK BAKELIKE GREEN',
 'PANDA AND BUNNIES STICKER SHEET',
 'STARS GIFT TAPE',
 'INFLATABLE PO

## 2) Vamos contar quantas vezes cada item aparece neste dataset.

In [68]:
item_freq = mba.groupby("Itemname")["Itemname"].count().sort_values(ascending=False)
item_freq.head(40)

Itemname
WHITE HANGING HEART T-LIGHT HOLDER    2269
JUMBO BAG RED RETROSPOT               2087
REGENCY CAKESTAND 3 TIER              1930
PARTY BUNTING                         1677
LUNCH BAG RED RETROSPOT               1570
ASSORTED COLOUR BIRD ORNAMENT         1465
SET OF 3 CAKE TINS PANTRY DESIGN      1360
PACK OF 72 RETROSPOT CAKE CASES       1328
LUNCH BAG  BLACK SKULL.               1315
NATURAL SLATE HEART CHALKBOARD        1246
JUMBO BAG PINK POLKADOT               1231
HEART OF WICKER SMALL                 1206
JUMBO STORAGE BAG SUKI                1191
PAPER CHAIN KIT 50'S CHRISTMAS        1183
JUMBO SHOPPER VINTAGE RED PAISLEY     1181
LUNCH BAG SPACEBOY DESIGN             1169
LUNCH BAG CARS BLUE                   1156
SPOTTY BUNTING                        1133
JAM MAKING SET PRINTED                1123
LUNCH BAG SUKI DESIGN                 1112
WOODEN PICTURE FRAME WHITE FINISH     1103
RECIPE BOX PANTRY YELLOW DESIGN       1101
LUNCH BAG PINK POLKADOT               1100
JA

In [69]:
item_freq = mba.groupby("Itemname")["Itemname"].count().sort_values(ascending=False)
item_freq.tail(20)

Itemname
PINK DOLLY HAIR CLIPS                  1
PINK CRYSTAL GUITAR PHONE CHARM        1
PINK CHRYSANTHEMUMS ART FLOWER         1
PINK CHERRY LIGHTS                     1
PINK CHERRY BLOSSOM CUP & SAUCER       1
PINK BOUDOIR T-LIGHT HOLDER            1
PINK BERTIE MOBILE PHONE CHARM         1
PINK BEADS+HAND PHONE CHARM            1
PINK BAROQUE FLOCK CANDLE HOLDER       1
PINK ACRYLIC JEWEL SNOWFLAKE           1
PEARL & SHELL 42"NECKL. IVORY          1
PAPER CRAFT , LITTLE BIRDIE            1
PAINTED HEART WREATH WITH BELL         1
PACK 4 FLOWER/BUTTERFLY PATCHES        1
ORANGE/FUSCHIA STONES NECKLACE         1
ORANGE PENDANT TRIPLE SHELL NECKLAC    1
ORANGE FELT VASE + FLOWERS             1
OPULENT VELVET SET/3 CANDLES           1
OOPS ! adjustment                      1
wrongly sold sets                      1
Name: Itemname, dtype: int64

### 3) Alguns itens são bem comuns, mas outros foram comprados uma única vez. Vamos fazer uma limpeza nesta dataset, caso contrário o tempo de execução do algoritmo será muito longo.

# Vamos manter apenas os itens que aparecem pelo menos 1000 vezes.

In [70]:
item_filt = item_freq[item_freq >= 1000].index.tolist()
len(item_filt) # agora poucos itens, 34 apenas, melhor para se trabalhar

34

In [71]:
item_filt

['WHITE HANGING HEART T-LIGHT HOLDER',
 'JUMBO BAG RED RETROSPOT',
 'REGENCY CAKESTAND 3 TIER',
 'PARTY BUNTING',
 'LUNCH BAG RED RETROSPOT',
 'ASSORTED COLOUR BIRD ORNAMENT',
 'SET OF 3 CAKE TINS PANTRY DESIGN',
 'PACK OF 72 RETROSPOT CAKE CASES',
 'LUNCH BAG  BLACK SKULL.',
 'NATURAL SLATE HEART CHALKBOARD',
 'JUMBO BAG PINK POLKADOT',
 'HEART OF WICKER SMALL',
 'JUMBO STORAGE BAG SUKI',
 "PAPER CHAIN KIT 50'S CHRISTMAS",
 'JUMBO SHOPPER VINTAGE RED PAISLEY',
 'LUNCH BAG SPACEBOY DESIGN',
 'LUNCH BAG CARS BLUE',
 'SPOTTY BUNTING',
 'JAM MAKING SET PRINTED',
 'LUNCH BAG SUKI DESIGN',
 'WOODEN PICTURE FRAME WHITE FINISH',
 'RECIPE BOX PANTRY YELLOW DESIGN',
 'LUNCH BAG PINK POLKADOT',
 'JAM MAKING SET WITH JARS',
 'SET OF 4 PANTRY JELLY MOULDS',
 'POSTAGE',
 'LUNCH BAG APPLE DESIGN',
 'ALARM CLOCK BAKELIKE RED',
 'RED RETROSPOT CHARLOTTE BAG',
 'VICTORIAN GLASS HANGING T-LIGHT',
 'ROSES REGENCY TEACUP AND SAUCER',
 'LUNCH BAG WOODLAND',
 'RABBIT NIGHT LIGHT',
 'GREEN REGENCY TEACUP AND

A próxima célula mantém apenas as transações que contém os 34 itens selecionados.

In [72]:
mba_filt = mba[mba["Itemname"].isin(item_filt)]
mba_filt

,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01.12.2010 08:26,"2,55",17850.0,United Kingdom
9,536367,ASSORTED COLOUR BIRD ORNAMENT,32,01.12.2010 08:34,"1,69",13047.0,United Kingdom
21,536368,JAM MAKING SET WITH JARS,6,01.12.2010 08:34,"4,25",13047.0,United Kingdom
27,536370,ALARM CLOCK BAKELIKE RED,24,01.12.2010 08:45,"3,75",12583.0,France
45,536370,POSTAGE,3,01.12.2010 08:45,18,12583.0,France
...,...,...,...,...,...,...,...
522026,581585,VICTORIAN GLASS HANGING T-LIGHT,12,09.12.2011 12:31,"1,95",15804.0,United Kingdom
522029,581585,RABBIT NIGHT LIGHT,12,09.12.2011 12:31,"2,08",15804.0,United Kingdom
522030,581585,ASSORTED COLOUR BIRD ORNAMENT,16,09.12.2011 12:31,"1,69",15804.0,United Kingdom
522035,581585,ALARM CLOCK BAKELIKE RED,4,09.12.2011 12:31,"3,75",15804.0,United Kingdom


Agora vamos ver quantas compras aparecem, agregando pelo atributo `BillNo`. Neste caso, já estamos também selecionando os itens de cada transação, e formatando como lista.

In [73]:
# BillNo, é o número único da compra

In [74]:
baskets = mba_filt.groupby("BillNo")["Itemname"].agg(list).tolist()
len(baskets)

12788

In [75]:
baskets

[['WHITE HANGING HEART T-LIGHT HOLDER'],
 ['ASSORTED COLOUR BIRD ORNAMENT'],
 ['JAM MAKING SET WITH JARS'],
 ['ALARM CLOCK BAKELIKE RED', 'POSTAGE'],
 ["PAPER CHAIN KIT 50'S CHRISTMAS"],
 ['WHITE HANGING HEART T-LIGHT HOLDER', 'WOODEN PICTURE FRAME WHITE FINISH'],
 ['WHITE HANGING HEART T-LIGHT HOLDER', 'WOODEN PICTURE FRAME WHITE FINISH'],
 ['JUMBO BAG PINK POLKADOT',
  'LUNCH BAG RED RETROSPOT',
  'PACK OF 72 RETROSPOT CAKE CASES',
  'JUMBO STORAGE BAG SUKI'],
 ['JAM MAKING SET PRINTED'],
 ['JUMBO SHOPPER VINTAGE RED PAISLEY', "PAPER CHAIN KIT 50'S CHRISTMAS"],
 ['JUMBO SHOPPER VINTAGE RED PAISLEY'],
 ['NATURAL SLATE HEART CHALKBOARD', 'HEART OF WICKER SMALL'],
 ['JAM MAKING SET PRINTED', 'JAM MAKING SET WITH JARS'],
 ['JUMBO BAG RED RETROSPOT'],
 ['HEART OF WICKER SMALL'],
 ['ALARM CLOCK BAKELIKE RED'],
 ['JAM MAKING SET WITH JARS',
  'JAM MAKING SET PRINTED',
  'WHITE HANGING HEART T-LIGHT HOLDER',
  'JUMBO BAG RED RETROSPOT'],
 ['ASSORTED COLOUR BIRD ORNAMENT'],
 ['WHITE HANGING H

Vamos inspecionar as duas primeiras compras.

In [76]:
baskets[:2]

[['WHITE HANGING HEART T-LIGHT HOLDER'], ['ASSORTED COLOUR BIRD ORNAMENT']]

Veja que elas correspondem às primeiras duas linhas do DataFrame acima.

Por fim, precisamos ordenar esta lista em ordem decrescente de itens, para permitir sua importação com o Pandas mais tarde.

In [77]:
baskets = sorted(baskets, key=len, reverse=True)

In [78]:
baskets 

[['RED RETROSPOT CHARLOTTE BAG',
  'LUNCH BAG RED RETROSPOT',
  'LUNCH BAG WOODLAND',
  'LUNCH BAG  BLACK SKULL.',
  'LUNCH BAG CARS BLUE',
  'PACK OF 72 RETROSPOT CAKE CASES',
  'JUMBO STORAGE BAG SUKI',
  "PAPER CHAIN KIT 50'S CHRISTMAS",
  'VICTORIAN GLASS HANGING T-LIGHT',
  'LUNCH BAG SPACEBOY DESIGN',
  'LUNCH BAG SUKI DESIGN',
  'LUNCH BAG PINK POLKADOT',
  'JUMBO BAG PINK POLKADOT',
  'JUMBO SHOPPER VINTAGE RED PAISLEY',
  'REGENCY CAKESTAND 3 TIER',
  'NATURAL SLATE HEART CHALKBOARD',
  'HEART OF WICKER SMALL',
  'RECIPE BOX PANTRY YELLOW DESIGN',
  'GREEN REGENCY TEACUP AND SAUCER',
  'ROSES REGENCY TEACUP AND SAUCER',
  'SET OF 3 CAKE TINS PANTRY DESIGN',
  'ALARM CLOCK BAKELIKE RED',
  'JAM MAKING SET WITH JARS',
  'JAM MAKING SET PRINTED',
  'SET OF 4 PANTRY JELLY MOULDS',
  'RABBIT NIGHT LIGHT',
  'LUNCH BAG APPLE DESIGN',
  'SPOTTY BUNTING',
  'PARTY BUNTING',
  'WOODEN PICTURE FRAME WHITE FINISH',
  'JUMBO BAG RED RETROSPOT',
  'WHITE HANGING HEART T-LIGHT HOLDER'],
 ['

### Vamos inspecionar de novo as duas primeiras compras.

In [79]:
baskets[:2] # duas compras, com os maiores itens comprados 

[['RED RETROSPOT CHARLOTTE BAG',
  'LUNCH BAG RED RETROSPOT',
  'LUNCH BAG WOODLAND',
  'LUNCH BAG  BLACK SKULL.',
  'LUNCH BAG CARS BLUE',
  'PACK OF 72 RETROSPOT CAKE CASES',
  'JUMBO STORAGE BAG SUKI',
  "PAPER CHAIN KIT 50'S CHRISTMAS",
  'VICTORIAN GLASS HANGING T-LIGHT',
  'LUNCH BAG SPACEBOY DESIGN',
  'LUNCH BAG SUKI DESIGN',
  'LUNCH BAG PINK POLKADOT',
  'JUMBO BAG PINK POLKADOT',
  'JUMBO SHOPPER VINTAGE RED PAISLEY',
  'REGENCY CAKESTAND 3 TIER',
  'NATURAL SLATE HEART CHALKBOARD',
  'HEART OF WICKER SMALL',
  'RECIPE BOX PANTRY YELLOW DESIGN',
  'GREEN REGENCY TEACUP AND SAUCER',
  'ROSES REGENCY TEACUP AND SAUCER',
  'SET OF 3 CAKE TINS PANTRY DESIGN',
  'ALARM CLOCK BAKELIKE RED',
  'JAM MAKING SET WITH JARS',
  'JAM MAKING SET PRINTED',
  'SET OF 4 PANTRY JELLY MOULDS',
  'RABBIT NIGHT LIGHT',
  'LUNCH BAG APPLE DESIGN',
  'SPOTTY BUNTING',
  'PARTY BUNTING',
  'WOODEN PICTURE FRAME WHITE FINISH',
  'JUMBO BAG RED RETROSPOT',
  'WHITE HANGING HEART T-LIGHT HOLDER'],
 ['

E as duas últimas.

In [80]:
baskets[-2:]

[['POSTAGE'], ['JUMBO BAG PINK POLKADOT']]

## 4) Agora podemos salvar essas compras em um arquivo txt na pasta do Drive. Para evitar confusão com o delimitador, já que um item pode conter vírgulas em sua descrição, vamos manter o caractere `;`.

In [81]:
#na página atual
DRIVE_DIRECTORY=''
with open(os.path.join(DRIVE_DIRECTORY, "itemlist.txt"), "w") as f:
    for basket in baskets:
        f.write(";".join(basket) + "\n")

Observe como os arquivos foram salvos.

In [82]:
caminho_pasta = os.getcwd()
#print(caminho_pasta)  # Pasta onde o notebook está rodando

In [83]:
!head {caminho_pasta}/itemlist.txt

RED RETROSPOT CHARLOTTE BAG;LUNCH BAG RED RETROSPOT;LUNCH BAG WOODLAND;LUNCH BAG  BLACK SKULL.;LUNCH BAG CARS BLUE;PACK OF 72 RETROSPOT CAKE CASES;JUMBO STORAGE BAG SUKI;PAPER CHAIN KIT 50'S CHRISTMAS;VICTORIAN GLASS HANGING T-LIGHT;LUNCH BAG SPACEBOY DESIGN;LUNCH BAG SUKI DESIGN;LUNCH BAG PINK POLKADOT;JUMBO BAG PINK POLKADOT;JUMBO SHOPPER VINTAGE RED PAISLEY;REGENCY CAKESTAND 3 TIER;NATURAL SLATE HEART CHALKBOARD;HEART OF WICKER SMALL;RECIPE BOX PANTRY YELLOW DESIGN;GREEN REGENCY TEACUP AND SAUCER;ROSES REGENCY TEACUP AND SAUCER;SET OF 3 CAKE TINS PANTRY DESIGN;ALARM CLOCK BAKELIKE RED;JAM MAKING SET WITH JARS;JAM MAKING SET PRINTED;SET OF 4 PANTRY JELLY MOULDS;RABBIT NIGHT LIGHT;LUNCH BAG APPLE DESIGN;SPOTTY BUNTING;PARTY BUNTING;WOODEN PICTURE FRAME WHITE FINISH;JUMBO BAG RED RETROSPOT;WHITE HANGING HEART T-LIGHT HOLDER
PACK OF 72 RETROSPOT CAKE CASES;JUMBO STORAGE BAG SUKI;PAPER CHAIN KIT 50'S CHRISTMAS;VICTORIAN GLASS HANGING T-LIGHT;LUNCH BAG SUKI DESIGN;LUNCH BAG PINK POLKADOT;

# 5) ir para a análise Apriori

## Algoritmo apriori

Carregue a lista de compras como um DataFrame do Pandas, lembre-se de selecionar o delimitador adequado.

In [84]:
base = pd.read_csv(os.path.join(DRIVE_DIRECTORY, "itemlist.txt"), delimiter=";", header=None)
base.tail(20)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
12768,POSTAGE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12769,ASSORTED COLOUR BIRD ORNAMENT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12770,LUNCH BAG SPACEBOY DESIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12771,NATURAL SLATE HEART CHALKBOARD,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12772,HEART OF WICKER SMALL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12773,PACK OF 72 RETROSPOT CAKE CASES,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12774,PAPER CHAIN KIT 50'S CHRISTMAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12775,PAPER CHAIN KIT 50'S CHRISTMAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12776,ASSORTED COLOUR BIRD ORNAMENT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12777,LUNCH BAG CARS BLUE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
base.head(20)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,RED RETROSPOT CHARLOTTE BAG,LUNCH BAG RED RETROSPOT,LUNCH BAG WOODLAND,LUNCH BAG BLACK SKULL.,LUNCH BAG CARS BLUE,PACK OF 72 RETROSPOT CAKE CASES,JUMBO STORAGE BAG SUKI,PAPER CHAIN KIT 50'S CHRISTMAS,VICTORIAN GLASS HANGING T-LIGHT,LUNCH BAG SPACEBOY DESIGN,...,JAM MAKING SET WITH JARS,JAM MAKING SET PRINTED,SET OF 4 PANTRY JELLY MOULDS,RABBIT NIGHT LIGHT,LUNCH BAG APPLE DESIGN,SPOTTY BUNTING,PARTY BUNTING,WOODEN PICTURE FRAME WHITE FINISH,JUMBO BAG RED RETROSPOT,WHITE HANGING HEART T-LIGHT HOLDER
1,PACK OF 72 RETROSPOT CAKE CASES,JUMBO STORAGE BAG SUKI,PAPER CHAIN KIT 50'S CHRISTMAS,VICTORIAN GLASS HANGING T-LIGHT,LUNCH BAG SUKI DESIGN,LUNCH BAG PINK POLKADOT,JUMBO SHOPPER VINTAGE RED PAISLEY,REGENCY CAKESTAND 3 TIER,NATURAL SLATE HEART CHALKBOARD,HEART OF WICKER SMALL,...,WHITE HANGING HEART T-LIGHT HOLDER,RED RETROSPOT CHARLOTTE BAG,LUNCH BAG RED RETROSPOT,LUNCH BAG WOODLAND,LUNCH BAG BLACK SKULL.,NaN,NaN,NaN,NaN,NaN
2,PACK OF 72 RETROSPOT CAKE CASES,JUMBO STORAGE BAG SUKI,PAPER CHAIN KIT 50'S CHRISTMAS,VICTORIAN GLASS HANGING T-LIGHT,LUNCH BAG SPACEBOY DESIGN,LUNCH BAG SUKI DESIGN,LUNCH BAG PINK POLKADOT,REGENCY CAKESTAND 3 TIER,NATURAL SLATE HEART CHALKBOARD,RECIPE BOX PANTRY YELLOW DESIGN,...,WHITE HANGING HEART T-LIGHT HOLDER,LUNCH BAG RED RETROSPOT,LUNCH BAG WOODLAND,LUNCH BAG BLACK SKULL.,LUNCH BAG CARS BLUE,NaN,NaN,NaN,NaN,NaN
3,ALARM CLOCK BAKELIKE RED,JAM MAKING SET WITH JARS,JAM MAKING SET PRINTED,SET OF 4 PANTRY JELLY MOULDS,RABBIT NIGHT LIGHT,LUNCH BAG APPLE DESIGN,JUMBO BAG RED RETROSPOT,WHITE HANGING HEART T-LIGHT HOLDER,RED RETROSPOT CHARLOTTE BAG,LUNCH BAG RED RETROSPOT,...,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,SET OF 3 CAKE TINS PANTRY DESIGN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,RED RETROSPOT CHARLOTTE BAG,LUNCH BAG WOODLAND,LUNCH BAG BLACK SKULL.,LUNCH BAG CARS BLUE,PACK OF 72 RETROSPOT CAKE CASES,JUMBO STORAGE BAG SUKI,PAPER CHAIN KIT 50'S CHRISTMAS,VICTORIAN GLASS HANGING T-LIGHT,LUNCH BAG SPACEBOY DESIGN,LUNCH BAG SUKI DESIGN,...,SPOTTY BUNTING,PARTY BUNTING,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,LUNCH BAG WOODLAND,LUNCH BAG BLACK SKULL.,PACK OF 72 RETROSPOT CAKE CASES,JUMBO STORAGE BAG SUKI,PAPER CHAIN KIT 50'S CHRISTMAS,VICTORIAN GLASS HANGING T-LIGHT,LUNCH BAG SUKI DESIGN,JUMBO SHOPPER VINTAGE RED PAISLEY,REGENCY CAKESTAND 3 TIER,NATURAL SLATE HEART CHALKBOARD,...,JUMBO BAG RED RETROSPOT,WHITE HANGING HEART T-LIGHT HOLDER,LUNCH BAG CARS BLUE,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,RECIPE BOX PANTRY YELLOW DESIGN,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,SET OF 3 CAKE TINS PANTRY DESIGN,ALARM CLOCK BAKELIKE RED,JAM MAKING SET WITH JARS,JAM MAKING SET PRINTED,SET OF 4 PANTRY JELLY MOULDS,RABBIT NIGHT LIGHT,LUNCH BAG APPLE DESIGN,...,NATURAL SLATE HEART CHALKBOARD,JUMBO BAG RED RETROSPOT,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,PAPER CHAIN KIT 50'S CHRISTMAS,VICTORIAN GLASS HANGING T-LIGHT,LUNCH BAG SPACEBOY DESIGN,LUNCH BAG PINK POLKADOT,REGENCY CAKESTAND 3 TIER,NATURAL SLATE HEART CHALKBOARD,HEART OF WICKER SMALL,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,JAM MAKING SET WITH JARS,...,PACK OF 72 RETROSPOT CAKE CASES,JUMBO STORAGE BAG SUKI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,PACK OF 72 RETROSPOT CAKE CASES,PAPER CHAIN KIT 50'S CHRISTMAS,VICTORIAN GLASS HANGING T-LIGHT,LUNCH BAG SPACEBOY DESIGN,LUNCH BAG SUKI DESIGN,LUNCH BAG PINK POLKADOT,JUMBO SHOPPER VINTAGE RED PAISLEY,REGENCY CAKESTAND 3 TIER,NATURAL SLATE HEART CHALKBOARD,HEART OF WICKER SMALL,...,LUNCH BAG WOODLAND,LUNCH BAG BLACK SKULL.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,PAPER CHAIN KIT 50'S CHRISTMAS,VICTORIAN GLASS HANGING T-LIGHT,LUNCH BAG SPACEBOY DESIGN,JUMBO BAG PINK POLKADOT,JUMBO SHOPPER VINTAGE RED PAISLEY,REGENCY CAKESTAND 3 TIER,NATURAL SLATE HEART CHALKBOARD,HEART OF WICKER SMALL,RECIPE BOX PANTRY YELLOW DESIGN,GREEN REGENCY TEACUP AND SAUCER,...,LUNCH BAG CARS BLUE,PACK OF 72 RETR

In [86]:
# notamos que o máximo de itens e uma única compra é de 32 compras em um único cliente. 

In [87]:
print(f'Compras selecionadas de cada cliente ou cada transação foram de {len(base)}.')

Compras selecionadas de cada cliente ou cada transação foram de 12788.


### Desta vez, você pode facilitar o processamento preenchendo os campos vazios com a string `nan`:

In [88]:
base.isnull().sum()

0         0
1      4437
2      7094
3      8771
4      9932
5     10713
6     11265
7     11602
8     11835
9     12041
10    12198
11    12321
12    12410
13    12501
14    12565
15    12623
16    12663
17    12700
18    12716
19    12736
20    12752
21    12763
22    12766
23    12773
24    12781
25    12785
26    12785
27    12787
28    12787
29    12787
30    12787
31    12787
dtype: int64

In [90]:
base.fillna("nan", inplace=True)

In [92]:
base.tail(1)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
12787,JUMBO BAG PINK POLKADOT,nan,nan,nan,nan,nan,nan,nan,nan,nan,...,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan


#### Agora converta o dataframe em uma lista de listas, primeiro chamando o atributo `values`, depois o método `tolist`:

In [93]:
transactions = base.values.tolist()

In [96]:
transactions[:1]

[['RED RETROSPOT CHARLOTTE BAG',
  'LUNCH BAG RED RETROSPOT',
  'LUNCH BAG WOODLAND',
  'LUNCH BAG  BLACK SKULL.',
  'LUNCH BAG CARS BLUE',
  'PACK OF 72 RETROSPOT CAKE CASES',
  'JUMBO STORAGE BAG SUKI',
  "PAPER CHAIN KIT 50'S CHRISTMAS",
  'VICTORIAN GLASS HANGING T-LIGHT',
  'LUNCH BAG SPACEBOY DESIGN',
  'LUNCH BAG SUKI DESIGN',
  'LUNCH BAG PINK POLKADOT',
  'JUMBO BAG PINK POLKADOT',
  'JUMBO SHOPPER VINTAGE RED PAISLEY',
  'REGENCY CAKESTAND 3 TIER',
  'NATURAL SLATE HEART CHALKBOARD',
  'HEART OF WICKER SMALL',
  'RECIPE BOX PANTRY YELLOW DESIGN',
  'GREEN REGENCY TEACUP AND SAUCER',
  'ROSES REGENCY TEACUP AND SAUCER',
  'SET OF 3 CAKE TINS PANTRY DESIGN',
  'ALARM CLOCK BAKELIKE RED',
  'JAM MAKING SET WITH JARS',
  'JAM MAKING SET PRINTED',
  'SET OF 4 PANTRY JELLY MOULDS',
  'RABBIT NIGHT LIGHT',
  'LUNCH BAG APPLE DESIGN',
  'SPOTTY BUNTING',
  'PARTY BUNTING',
  'WOODEN PICTURE FRAME WHITE FINISH',
  'JUMBO BAG RED RETROSPOT',
  'WHITE HANGING HEART T-LIGHT HOLDER']]

#### Por curiosidade, inspecione a transação de índice 300.

In [98]:
transactions[300] # produtos compraoos

['PARTY BUNTING',
 'REGENCY CAKESTAND 3 TIER',
 'SET OF 4 PANTRY JELLY MOULDS',
 'VICTORIAN GLASS HANGING T-LIGHT',
 'LUNCH BAG RED RETROSPOT',
 'LUNCH BAG CARS BLUE',
 'LUNCH BAG WOODLAND',
 'LUNCH BAG  BLACK SKULL.',
 'LUNCH BAG SPACEBOY DESIGN',
 'LUNCH BAG SUKI DESIGN',
 'LUNCH BAG PINK POLKADOT',
 'WHITE HANGING HEART T-LIGHT HOLDER',
 'PACK OF 72 RETROSPOT CAKE CASES',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan',
 'nan']

### Agora invoque o algoritmo `apriori`, informando as transações, `min_support=0.02`, `min_confidence=0.25` e `min_lift=5`.

In [99]:

n_vezes_vendas = 4
semana_dia = 7

compras = n_vezes_vendas*semana_dia

print('compras',compras)

qtd_pessoas_comprou_local = len(base)
print('qtd. de pessoas que compraram no local',qtd_pessoas_comprou_local)

print('suporte')
# divisão de produtos vendidos/ quantidades totais de produtos 
print(compras/qtd_pessoas_comprou_local) # valor que é a base para o suporte na regra apriori
# valor os produtos que são vendidos 4 vezes ao dia, na nossa base de dados

compras 28
qtd. de pessoas que compraram no local 12788
suporte
0.002189552705661558


In [101]:
rules = apriori(transactions, min_support=0.02, min_confidence=0.25, min_lift=5)

### Execute o algoritmo, convertendo as regras para uma lista.

In [102]:
results = list(rules)

Cheque quantas regras foram geradas.

In [103]:
len(results)

78

In [107]:
import prettyprinter as pp
pp.pprint(results)
print('\nquantidade de dados:', len(resultados)) # quantidade de dados diminuiram

[
    apyori.RelationRecord(
        items=frozenset([
            'ROSES REGENCY TEACUP AND SAUCER',
            'GREEN REGENCY TEACUP AND SAUCER'
        ]),
        support=0.057162965279949955,
        ordered_statistics=[
            apyori.OrderedStatistic(
                items_base=frozenset(['GREEN REGENCY TEACUP AND SAUCER']),
                items_add=frozenset(['ROSES REGENCY TEACUP AND SAUCER']),
                confidence=0.7497435897435898,
                lift=9.464680183258665
            ),
            apyori.OrderedStatistic(
                items_base=frozenset(['ROSES REGENCY TEACUP AND SAUCER']),
                items_add=frozenset(['GREEN REGENCY TEACUP AND SAUCER']),
                confidence=0.721618953603159,
                lift=9.464680183258663
            )
        ]
    ),
    apyori.RelationRecord(
        items=frozenset([
            'LUNCH BAG  BLACK SKULL.',
            'LUNCH BAG PINK POLKADOT'
        ]),
        support=0.04183609634031905,
     

NameError: name 'resultados' is not defined

In [110]:
print(results[0])
print()
print(results[0][0])
print()
print(results[0][1])
print()
print(results[0][2])


RelationRecord(items=frozenset({'ROSES REGENCY TEACUP AND SAUCER', 'GREEN REGENCY TEACUP AND SAUCER'}), support=0.057162965279949955, ordered_statistics=[OrderedStatistic(items_base=frozenset({'GREEN REGENCY TEACUP AND SAUCER'}), items_add=frozenset({'ROSES REGENCY TEACUP AND SAUCER'}), confidence=0.7497435897435898, lift=9.464680183258665), OrderedStatistic(items_base=frozenset({'ROSES REGENCY TEACUP AND SAUCER'}), items_add=frozenset({'GREEN REGENCY TEACUP AND SAUCER'}), confidence=0.721618953603159, lift=9.464680183258663)])

frozenset({'ROSES REGENCY TEACUP AND SAUCER', 'GREEN REGENCY TEACUP AND SAUCER'})

0.057162965279949955

[OrderedStatistic(items_base=frozenset({'GREEN REGENCY TEACUP AND SAUCER'}), items_add=frozenset({'ROSES REGENCY TEACUP AND SAUCER'}), confidence=0.7497435897435898, lift=9.464680183258665), OrderedStatistic(items_base=frozenset({'ROSES REGENCY TEACUP AND SAUCER'}), items_add=frozenset({'GREEN REGENCY TEACUP AND SAUCER'}), confidence=0.721618953603159, lift=

In [111]:
print(results[0][2][0][0]) # primeira regra "se"
print()
print(results[0][2][0][1]) # segunda regra "então"
print()
print(results[0][2][0][2]) # confiança
print()
print(results[0][2][0][3]) # valores do lift

frozenset({'GREEN REGENCY TEACUP AND SAUCER'})

frozenset({'ROSES REGENCY TEACUP AND SAUCER'})

0.7497435897435898

9.464680183258665


## 6) Utilize o mesmo código de parsing apresentado em aula para organizar os resultados em um DataFrame.

In [112]:
A = []
B = []
support = []
confidence = []
lift = []

for result in results:
    s = result[1]
    result_rules = result[2]
    for result_rule in result_rules:
        a = list(result_rule[0])
        b = list(result_rule[1])
        c = result_rule[2]
        l = result_rule[3]
        A.append(a)
        B.append(b)
        support.append(s)
        confidence.append(c)
        lift.append(l)

In [114]:
rules_df = pd.DataFrame({'A': A, 'B': B, 'support': support, 'confidence': confidence, 'lift': lift})

In [115]:
rules_df.tail()

,A,B,support,confidence,lift
480,"[JUMBO STORAGE BAG SUKI, nan, JUMBO SHOPPER VI...","[JUMBO BAG RED RETROSPOT, JUMBO BAG PINK POLKA...",0.020801,0.512524,8.002635
481,"[JUMBO BAG RED RETROSPOT, nan, JUMBO SHOPPER V...",[JUMBO STORAGE BAG SUKI],0.020801,0.707447,7.679822
482,"[JUMBO STORAGE BAG SUKI, JUMBO BAG RED RETROSP...",[JUMBO SHOPPER VINTAGE RED PAISLEY],0.020801,0.647202,7.098129
483,"[JUMBO STORAGE BAG SUKI, nan, JUMBO SHOPPER VI...",[JUMBO BAG RED RETROSPOT],0.020801,0.866450,5.368293
484,"[JUMBO STORAGE BAG SUKI, JUMBO BAG RED RETROSP...",[JUMBO BAG PINK POLKADOT],0.020801,0.687339,7.258204


# 7) Exiba o DataFrame com os resultados, ordenado de forma decrescente por valor de lift.

In [116]:
rules_df.sort_values(by = 'lift', ascending = False).tail(25)

,A,B,support,confidence,lift
106,"[LUNCH BAG SUKI DESIGN, nan]",[LUNCH BAG APPLE DESIGN],0.034720,0.412256,5.158447
393,"[LUNCH BAG CARS BLUE, LUNCH BAG WOODLAND, nan]",[LUNCH BAG RED RETROSPOT],0.021270,0.621005,5.153411
389,"[LUNCH BAG CARS BLUE, LUNCH BAG WOODLAND]","[LUNCH BAG RED RETROSPOT, nan]",0.021270,0.619590,5.145011
79,"[LUNCH BAG SPACEBOY DESIGN, LUNCH BAG BLACK S...",[LUNCH BAG RED RETROSPOT],0.024554,0.619329,5.139510
305,"[LUNCH BAG SPACEBOY DESIGN, LUNCH BAG BLACK S...",[LUNCH BAG RED RETROSPOT],0.024476,0.618577,5.133266
301,"[LUNCH BAG SPACEBOY DESIGN, LUNCH BAG BLACK S...","[LUNCH BAG RED RETROSPOT, nan]",0.024476,0.617357,5.126468
52,"[LUNCH BAG CARS BLUE, LUNCH BAG BLACK SKULL.]",[LUNCH BAG RED RETROSPOT],0.024945,0.615830,5.110471
260,"[LUNCH BAG CARS BLUE, LUNCH BAG BLACK SKULL.,...",[LUNCH BAG RED RETROSPOT],0.024867,0.615087,5.104304
255,"[LUNCH BAG CARS BLUE, LUNCH BAG BLACK SKULL.]","[LUNCH BAG RED RETROSPOT, nan]",0.024867,0.613900,5.097759
3,[LUNCH BAG PINK POLKADOT],[LUNCH BAG BLACK SKULL.],0.041836,0.498602,5.060415


In [117]:
rules_df.sort_values(by = 'lift', ascending = False).head(5)

,A,B,support,confidence,lift
13,"[ROSES REGENCY TEACUP AND SAUCER, REGENCY CAKE...",[GREEN REGENCY TEACUP AND SAUCER],0.029715,0.777096,10.192313
10,[GREEN REGENCY TEACUP AND SAUCER],"[ROSES REGENCY TEACUP AND SAUCER, REGENCY CAKE...",0.029715,0.389744,10.192313
180,"[ROSES REGENCY TEACUP AND SAUCER, REGENCY CAKE...",[GREEN REGENCY TEACUP AND SAUCER],0.029637,0.776639,10.186322
173,[GREEN REGENCY TEACUP AND SAUCER],"[ROSES REGENCY TEACUP AND SAUCER, REGENCY CAKE...",0.029637,0.388718,10.186322
177,"[ROSES REGENCY TEACUP AND SAUCER, REGENCY CAKE...","[GREEN REGENCY TEACUP AND SAUCER, nan]",0.029637,0.775051,10.175928
